# ליגת העל — שחקנים שצברו לפחות 10 שערים + בישולים בעונה

נתוני מקור: קבצי Transfermarkt המאוחדים (`top_scorers_all_seasons_*`, `top_assists_all_seasons_*`) מתיקיית `data/raw/player_stats/`.

**חישוב:** לכל שחקן–קבוצה–עונה מאחדים שערים מטבלת המבקיעים ובישולים מטבלת המבשילים, ומסכמים `G+A`. סף: **10 ומעלה**.

**"מקבוצות שונות":** בכל עונה נספור כמה שחקנים עמדו בסף (`players_at_least_10_ga`), ומכמה **קבוצות שונות** יש לפחות שחקן כזה (`distinct_clubs`). יש גם פירוט לפי קבוצה בטבלה הבאה.

**הערת נתונים:** לפני המיזוג מרככים כפילויות בטבלת הבישולים (בעיקר עונת 2009/10 בקובץ האוחד), ומשתמשים ב-`groupby(..., dropna=False)` כדי לא לאבד שורות עם `club` חסר.

In [2]:
from pathlib import Path

import pandas as pd


def find_notebooks_root() -> Path:
    p = Path.cwd()
    for _ in range(8):
        if (p / "data" / "raw" / "player_stats").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Could not find data/raw/player_stats under parents of cwd")


ROOT = find_notebooks_root()
PLAYER_STATS_DIR = ROOT / "data" / "raw" / "player_stats"
SCORERS_CSV = PLAYER_STATS_DIR / "top_scorers_all_seasons_ligat_haal_transfermarkt.csv"
ASSISTS_CSV = PLAYER_STATS_DIR / "top_assists_all_seasons_ligat_haal_transfermarkt.csv"

THRESHOLD = 10

print("ROOT:", ROOT)
print("Scorers:", SCORERS_CSV.exists(), SCORERS_CSV)
print("Assists:", ASSISTS_CSV.exists(), ASSISTS_CSV)

ROOT: c:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks
Scorers: True c:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks\data\raw\player_stats\top_scorers_all_seasons_ligat_haal_transfermarkt.csv
Assists: True c:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks\data\raw\player_stats\top_assists_all_seasons_ligat_haal_transfermarkt.csv


In [3]:
def load_merged_season_stats(scorers_path: Path, assists_path: Path) -> pd.DataFrame:
    key = ["player_url", "season_year", "club"]

    def _dedupe(df: pd.DataFrame, goals_c: str, ast_c: str) -> pd.DataFrame:
        d = df.copy()
        d[goals_c] = pd.to_numeric(d[goals_c], errors="coerce")
        d[ast_c] = pd.to_numeric(d[ast_c], errors="coerce")
        agg = {"season": "first", "player": "first", goals_c: "max", ast_c: "max"}
        if "rank" in d.columns:
            agg["rank"] = "min"
        # dropna=False — לא לאבד שורות עם club ריק (מיזוג חיצוני עדיין יחבר שחקנים מטבלת השערים)
        return d.groupby(key, as_index=False, dropna=False).agg(agg)

    s = _dedupe(
        pd.read_csv(scorers_path).rename(columns={"goals": "goals_sc", "assists": "assists_sc"}),
        "goals_sc",
        "assists_sc",
    )[["season", "season_year", "player", "player_url", "club", "goals_sc", "assists_sc"]]
    a = _dedupe(
        pd.read_csv(assists_path).rename(columns={"goals": "goals_as", "assists": "assists_as"}),
        "goals_as",
        "assists_as",
    )[["season", "season_year", "player", "player_url", "club", "goals_as", "assists_as"]]

    m = s.merge(a, on=key, how="outer", suffixes=("_s", "_a"))
    # שם ועונה עקביים
    m["player"] = m["player_s"].fillna(m["player_a"])
    m["season"] = m["season_s"].fillna(m["season_a"])
    m.drop(columns=["player_s", "player_a", "season_s", "season_a"], inplace=True)

    for col in ("goals_sc", "assists_sc", "goals_as", "assists_as"):
        m[col] = pd.to_numeric(m[col], errors="coerce")

    # כשיש את שני המקורות — לוקחים מקסימום (אמור להתאים); אחרת את הערך הקיים
    m["goals"] = m[["goals_sc", "goals_as"]].max(axis=1, skipna=True)
    m["assists"] = m[["assists_sc", "assists_as"]].max(axis=1, skipna=True)
    m["goals"] = m["goals"].fillna(0).astype(int)
    m["assists"] = m["assists"].fillna(0).astype(int)
    m["goals_plus_assists"] = m["goals"] + m["assists"]
    return m


full = load_merged_season_stats(SCORERS_CSV, ASSISTS_CSV)
full.head(10)

,season_year,player_url,club,goals_sc,assists_sc,goals_as,assists_as,player,season,goals,assists,goals_plus_assists
0,2023,https://www.transfermarkt.com/aaron-leya-iseka...,Hapoel Hadera,1.0,NaN,NaN,NaN,Aaron Leya Iseka,2023/24,1,0,1
1,2018,https://www.transfermarkt.com/aaron-olanare/pr...,Beitar Jerusalem,1.0,NaN,NaN,NaN,Aaron Olanare,2018/19,1,0,1
2,2015,https://www.transfermarkt.com/aaron-schoenfeld...,for 2 clubs,3.0,NaN,NaN,NaN,Aaron Schoenfeld,2015/16,3,0,3
3,2016,https://www.transfermarkt.com/aaron-schoenfeld...,for 2 clubs,2.0,NaN,NaN,NaN,Aaron Schoenfeld,2016/17,2,0,2
4,2018,https://www.transfermarkt.com/aaron-schoenfeld...,Maccabi Tel Aviv,1.0,NaN,NaN,1.0,Aaron Schoenfeld,2018/19,1,1,2
5,2019,https://www.transfermarkt.com/aaron-schoenfeld...,Maccabi Tel Aviv,1.0,NaN,NaN,NaN,Aaron Schoenfeld,2019/20,1,0,1
6,2006,https://www.transfermarkt.com/abbas-suan/profi...,Maccabi Haifa,1.0,NaN,NaN,1.0,Abbas Suan,2006/07,1,1,2
7,2007,https://www.transfermarkt.com/abbas-suan/profi...,Ironi Kiryat Shmona,3.0,NaN,NaN,NaN,Abbas Suan,2007/08,3,0,3
8,2009,https://www.transfermarkt.com/abbas-suan/profi...,Ihud Bnei Sakhnin,NaN,NaN,NaN,1.0,Abbas Suan,2009/10,0,1,1
9,2019,https://www.transfermarkt.com/abdallah-hleihel...,Ironi Kiryat Shmona,1.0,NaN,NaN,NaN,Abdallah Hleihel,2019/20,1,0,1


In [4]:
above = full[full["goals_plus_assists"] >= THRESHOLD].copy()

by_season = (
    above.groupby(["season", "season_year"], sort=True)
    .agg(
        players_at_least_10_ga=("player_url", "count"),
        distinct_clubs=("club", "nunique"),
    )
    .reset_index()
)

by_season

,season,season_year,players_at_least_10_ga,distinct_clubs
0,2006/07,2006,16,9
1,2007/08,2007,12,7
2,2008/09,2008,13,6
3,2009/10,2009,18,11
4,2010/11,2010,16,12
5,2011/12,2011,25,13
6,2012/13,2012,16,10
7,2013/14,2013,15,8
8,2014/15,2014,12,7
9,2015/16,2015,16,9


## פירוט לפי קבוצה בעונה (כמה שחקנים בקבוצה עברו את הסף)

In [5]:
by_club_season = (
    above.groupby(["season", "season_year", "club"])
    .size()
    .reset_index(name="players_ge_10")
    .sort_values(["season_year", "club"])
)

by_club_season.head(30)

,season,season_year,club,players_ge_10
0,2006/07,2006,Beitar Jerusalem,2
1,2006/07,2006,Bnei Yehuda Tel Aviv,1
2,2006/07,2006,FC Ashdod,3
3,2006/07,2006,Hapoel Kfar Saba,1
4,2006/07,2006,Hapoel Tel Aviv,3
5,2006/07,2006,Maccabi Haifa,2
6,2006/07,2006,Maccabi Herzliya,1
7,2006/07,2006,Maccabi Netanya,1
8,2006/07,2006,Maccabi Tel Aviv,2
9,2007/08,2007,Beitar Jerusalem,1


## בחירת עונה לטבלה מלאה

In [6]:
SEASON_YEAR_FILTER: int | None = None  # לדוגמה: 2024 לעונה 2024/25; או None לכל העונות יחד

show = above if SEASON_YEAR_FILTER is None else above[above["season_year"] == SEASON_YEAR_FILTER]
cols = ["season", "club", "player", "goals", "assists", "goals_plus_assists"]
show[cols].sort_values(["season", "goals_plus_assists", "goals"], ascending=[True, False, False])

,season,club,player,goals,assists,goals_plus_assists
3278,2006/07,FC Ashdod,Shay Holtsman,12,6,18
3724,2006/07,FC Ashdod,Yaniv Azran,15,2,17
3165,2006/07,Hapoel Tel Aviv,Salim Tuama,8,7,15
1969,2006/07,Bnei Yehuda Tel Aviv,Lior Asulin,13,1,14
2682,2006/07,Maccabi Herzliya,Omer Buksenbaum,12,1,13
...,...,...,...,...,...,...
2146,2025/26,Maccabi Netanya,Maor Levi,6,5,11
1875,2025/26,Maccabi Haifa,Kenji Gorré,4,7,11
1392,2025/26,Maccabi Haifa,Guy Melamed,9,1,10
49,2025/26,Beitar Jerusalem,Adi Yona,8,2,10
